# Concept Tutorial: Gates

In smfBursts, `smf.GateGroup` objects are used to filter rows in tables, usually for selecting populations of bursts.

These gates create boolean masks, and therefore it is possible to consider boolean operation between gates. 
These are fully supported in smfBursts.

## Initial imports/setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import smfbursts as smf

raw = smf.photonHDF5.load('HP3_TE300_SPC630.hdf5')
data = smf.photonHDF5.regularize_dets(raw)

prd = smf.Param(smf.Periods, detdef=data.detdef, period=60.0, start_at='time_min', stop_at='over')
bg = smf.Param(smf.BG, base=prd, func=smf.bg.exp_mlefit, tail_min=0.005)
brst = smf.Param(smf.Bursts, bg=bg, streams=smf.PhSel('0ex_1ex1em'), m=10, F=6.0)

nphdd = smf.Column(brst, 'nph_raw', smf.PhSel('0ex0em'))
nphda = smf.Column(brst, 'nph_raw', smf.PhSel('0ex1em'))
nphaa = smf.Column(brst, 'nph_raw', smf.PhSel('1ex1em'))
nphall = smf.Column(brst, 'nph_raw', smf.PhSel('all'))

trmtg = smf.Column(brst, 'mTdiff', (smf.PhSel('1ex1em'), smf.PhSel('0ex'))) # proxy for bleaching

FileNotFoundError: ``/home/paul/Python/fretbursts_new/docs/source/HP3_TE300_SPC630.hdf5`` does not exist

## Gate `make_..._gate` functions

The easiest way to create gates is using the `make_..._gate` functions, of which there are:

1. `smf.gates.make_lt_gate(column, mx)` make gate for values in the Column `column` that are less than the value `mx`
2. `smf.gates.make_geq_gate(column, mn)` make gate for values in Column `column` that are greater than or equal to the value `mn`
3. `smf.gates.make_ellipsoid_gate(colx, coly, cx=0, cy=0, dx=1.0, dy=1.0, theta=0.0)` all values inside ellipse (open bountary)
4. `smf.gates.make_inv_ellipsoid_gate(colx, coly, cx=0, cy=0, dx=1.0, dy=1.0, theta=0.0)` all values outside elipse (closed boundary)
5. `smf.gates.make_upper_inclusive_percentile_gate(column, up)` gate of values in upper percentile of column (non-atomic, closed boundary)
6. `smf.gates.make_lower_exclusive_percentile_gate` gate of values in lower percentile of column (non-atomic, open boundary)
7. `smf.gates.make_exclude_nan(col)` gate excludes NAN values from column. This is usually only used internally 

In [ ]:
g_nphdd30 = smf.make_geq_gate(nphdd, 30)

fig, ax = plt.subplots(1,2, figsize=(12,5))
bins=np.arange(0,200,10)
smf.plot.hist_stair(data, nphdd, gate=g_nphdd30, label='gated nphdd', bins=bins, ax=ax[0])
smf.plot.hist_stair(data, nphdd, label='ungated nphdd', bins=bins, ax=ax[0])
ax[0].set_title("gate on same column")
smf.plot.hist_stair(data, nphda, gate=g_nphdd30, label='gated nphdd', bins=bins, ax=ax[1])
smf.plot.hist_stair(data, nphda, label='ungated nphdd', bins=bins, ax=ax[1])
ax[1].set_title("gate on different column")
plt.legend()

## Gate Logical Operations

Logical operations are permitted between gates.

Note that these can be arbitrarily nested, the limit for nesting is determiend by the maximum number of dimensions in a numpy array.
(The reason for this will be explained later).

In [ ]:
g_nphda30 = smf.make_geq_gate(nphda, 30)

In [ ]:
# commutative operations
g_nphdd30_AND_nphda30 = g_nphdd30 & g_nphda30
g_nphdd30_OR_nphda30 = g_nphdd30 | g_nphda30
g_nphdd30_EQ_nphda30 = g_nphdd30 @ g_nphda30
g_nphdd30_XOR_nphda30 = g_nphdd30 ^ g_nphda30
# non-commutative operations
g_nphdd30_MINUS_nphda30 = g_nphdd30 - g_nphda30
g_nphdd30_IMPLIES_nphda30 = g_nphdd30 >> g_nphda30
g_nphdd30_RIMPLIES_nphda30 = g_nphdd30 << g_nphda30

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(10,10), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.7))

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30, s=1.0, ax=ax[0,0])
ax[0,0].set_title("Gate A")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30, s=1.0, ax=ax[0,1])
ax[0,1].set_title("Gate B")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_AND_nphda30, s=1.0, ax=ax[1,0])
ax[1,0].set_title(r"$A \cap B$ (&)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_OR_nphda30, s=1.0, ax=ax[1,1])
ax[1,1].set_title(r"$A \cup B$ (|)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_EQ_nphda30, s=1.0, ax=ax[2,0])
ax[2,0].set_title("$A = B$ (@)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_XOR_nphda30, s=1.0, ax=ax[2,1])
ax[2,1].set_title(r"$A \neq B$ (^)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_MINUS_nphda30, s=1.0, ax=ax[0,2])
ax[0,2].set_title(r"$A - B$ (-)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_IMPLIES_nphda30, s=1.0, ax=ax[1,2])
ax[1,2].set_title(r"$A \Longrightarrow B$ (>>)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_RIMPLIES_nphda30, s=1.0, ax=ax[2,2])
ax[2,2].set_title(r"$A \Longrightarrow B$ (<<)")

Note that some of these logical operators are cummutative, ie the order does not matter, while others are non-commutative:

### Examples of commutative operators

In [ ]:
# commutative operations
g_nphda30_AND_nphdd30 = g_nphda30 & g_nphdd30
g_nphda30_OR_nphdd30 = g_nphda30 | g_nphdd30
g_nphda30_EQ_nphdd30 = g_nphda30 @ g_nphdd30
g_nphda30_XOR_nphdd30 = g_nphda30 ^ g_nphdd30


fig, ax = plt.subplots(4,2, figsize=(6, 12), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.7))

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_AND_nphda30, s=1.0, ax=ax[0,0])
ax[0,0].set_title(r"$A \cap B$ (&)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_AND_nphdd30, s=1.0, ax=ax[0,1])
ax[0,1].set_title(r"$B \cap A$ (&)")

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_OR_nphda30, s=1.0, ax=ax[1,0])
ax[1,0].set_title(r"$A \cup B$ (|)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_OR_nphdd30, s=1.0, ax=ax[1,1])
ax[1,1].set_title(r"$B \cup A$ (|)")

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_EQ_nphda30, s=1.0, ax=ax[2,0])
ax[2,0].set_title(r"$A = B$ (@)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_EQ_nphdd30, s=1.0, ax=ax[2,1])
ax[2,1].set_title(r"$B = A$ (@)")

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_XOR_nphda30, s=1.0, ax=ax[3,0])
ax[3,0].set_title(r"$A \neq B$ (^)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_XOR_nphdd30, s=1.0, ax=ax[3,1])
ax[3,1].set_title(r"$B \neq A$ (^)")

### Examples of non-commutative operators

In [ ]:
# non-commutative operations
g_nphda30_MINUS_nphdd30 = g_nphda30 - g_nphdd30
g_nphda30_IMPLIES_nphdd30 = g_nphda30 >> g_nphdd30
g_nphda30_RIMPLIES_nphdd30 = g_nphda30 << g_nphdd30


fig, ax = plt.subplots(3,2, figsize=(6, 8), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.5))

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_MINUS_nphda30, s=1.0, ax=ax[0,0])
ax[0,0].set_title(r"$A - B$ (-)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_MINUS_nphdd30, s=1.0, ax=ax[0,1])
ax[0,1].set_title(r"$B - A$ (-)")

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_IMPLIES_nphda30, s=1.0, ax=ax[1,0])
ax[1,0].set_title(r"$A \Longrightarrow B$ (>>)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_IMPLIES_nphdd30, s=1.0, ax=ax[1,1])
ax[1,1].set_title(r"$B \Longrightarrow A$ (>>)")

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_RIMPLIES_nphda30, s=1.0, ax=ax[2,0])
ax[2,0].set_title(r"$A \Longleftarrow B$ (<<)")
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_RIMPLIES_nphdd30, s=1.0, ax=ax[2,1])
ax[2,1].set_title(r"$B \Longleftarrow A$ (<<)")

Also note that `A>>B` is equivalent to ``B<<A``

### Inversion

Gates can also be inverted:

In [ ]:
g_nphdd30_inv = ~g_nphdd30
g_nphda30_inv = ~g_nphda30


fig, ax = plt.subplots(2,2, figsize=(8,8), sharex=True, sharey=True, gridspec_kw=dict(hspace=0.5))

smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30, s=1.0, ax=ax[0,0])
ax[0,0].set_title(r'$A$')
smf.plot.scatter(data, nphdd, nphda, gate=g_nphdd30_inv, s=1.0, ax=ax[0,1])
ax[0,1].set_title(r'$\neg A$')
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30, s=1.0, ax=ax[1,0])
ax[1,0].set_title(r'$B$')
smf.plot.scatter(data, nphdd, nphda, gate=g_nphda30_inv, s=1.0, ax=ax[1,1])
ax[1,1].set_title(r'$\neg B$')

## GateGroup Composition

### Inspecting GateGroups

Now lets undersand how GateGroups work under the hood.

Most GateGroups have 2 important properties: 
1. Truthtable
2. Sequence of Gates

Lets look at the first gate we created: `g_nphdd30`

In [ ]:
print(g_nphdd30.truthtable)

for i, g in enumerate(g_nphdd30.gates):
    print(f'----------{i}-----------')
    print(g.description)
    print('-------------------------')

There is only 1 gate, so why do we have a truthtable?
Because in the truthtable was `[True, False]` we would have the inverse.
Indeed, if we examine the inverse, that is exactly what it is:

In [ ]:
print(g_nphdd30_inv.truthtable)

for i, g in enumerate(g_nphdd30_inv.gates):
    print(f'----------{i}-----------')
    print(g.description)
    print('-------------------------')

### All and None gates

The use of a truthtable as an array also means that it can even have a single-element 0-D array.

PhotonBursts intentionally uses this to represent "all" and "none" gates:

In [ ]:
g_none = smf.GateGroup(np.array(False)) # the None gate with no param
g_all = smf.GateGroup(np.array(True)) # the None gate with no param
g_none, g_all

The above gates can be applied to any Column or Param with the regate method, `g_none` will create an empty column/param,
while `g_all` is nearly equivalent to the `param.degate()` method.

> **Note**
> 
> For columns there is a "sharp corner" which is non-atomic columsn.
> For atomic columns (which is the vast majority) `col.regate(g_all)` is equivalent to `col.degate()`.
> However, for non-atomic columns (mainly the 'sep' column), 
> `col.degate()` sets the gate to the gate defining on which rows the column is computed (the base_gate of the param).
> While `col.regate(g_all)` causes the gate to indeed be all columns of the origin_param- and thus there will be many "filled" rows.

Specifically to handle the fact that the previously created `g_none` and `g_all` gates do not have a defined set of rows,
it is also possible to specify the `Param` for which a none/all gate is defined:

In [ ]:
g_none_b = smf.GateGroup(np.array(False), param=brst)
g_all_b = smf.GateGroup(np.array(True), param=brst)

The above `g_none_b` and `g_all_b` gates now only specifically work with columns/params based on `brst`.
`regate()` will work only with `Param` and `Column` objects based on `brst`.

It should be noted that the `base_gate` of an ungated `Param` is always an "all" gate.
Therefore, we can test equality between `g_all_b` with `brst.base_gate`, and get `True`.

Equality requires strictness, so the "general" all gate will evaluate as `False` when compared to any gate with a defined param.

In [ ]:
g_all_b == brst.base_gate, g_all == brst.base_gate

#### GateGroup "truthiness"

`GateGroup` objects are given a "truth" value so they can be used in `if` statements and the like.

The definition is simple: if the gate is anything but a "none" type gate, it is `True`, and "none" type gates, regarless of the param, are `False`

In [ ]:
bool(g_none), bool(g_none_b), bool(g_nphdd30), bool(g_all), bool(g_all_b)

Between the two, the gates parameter is the same, only the truthtable is inverted.

### The **AND** truthtable

Now, when we combine 2 GateGroups with a logical operation, we expand the truthtable to a 2D array:

In [ ]:
g_nphdd30_AND_nphda30.truthtable

In [ ]:
for i, g in enumerate(g_nphdd30_AND_nphda30.gates):
    print(f'----------{i}-----------')
    print(g.description)
    print('-------------------------')

### The `OR` truthtable:

In [ ]:
for i, g in enumerate(g_nphdd30_OR_nphda30.gates):
    print(f'----------{i}-----------')
    print(g.description)
    print('-------------------------')

The difference between all the binary logical operations is the truthtable, the gates object does not change between them.

So the Gate objects define some basic boolean masking operation on a set of columns,
while the GateGroup object is responsible for combining a set of Gates using the truthtable to create a final boolean mask.

Single Gate GateGroups are implemented for 2 reasons
1. All gating is therefore handled by GateGroups, simplifying getting operations
2. Through the inverse truthtable, there are fewer necessary types of Gate functions necesary to define.

## `Gate` objects in `GateGroup`s

While usually GateGroup objects are created through `make_..._gate` functions,
it is possible to create them directly from a truthtable and a set of Gates.

Of course, you then have to create the Gate itself first.

**Creating a simple Gate object**

The `Gate` object has 3 important components:
1. `Gate.gatedef`: GateDef (gating function)
2. `Gate.columns`: Sequence of `Column` objects
3. `Gate.params`: dictionary of params (keyword arguments to GateDef func)

In [ ]:
gate_nphdd30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphdd,), {'m':30.0, 'vec':np.array([1.0])})

Remember that since all instruction objects support the `==` operator to test if they represent the same instructions,
we can sheck if the above created gate is indeed the gate created from `make_geq_gate`

In [ ]:
g_nphdd30.gates[0] == gate_nphdd30

**Create a `GateGroup` from `Gate`s**

In order to create a `GateGroup` directly, we specify the following:
`GateGroup(truthtable, gate1, gate2, ...)`

In [ ]:
single_tt = np.array([False, True])
ng_nphdd30 = smf.GateGroup(single_tt, gate_nphdd30)

And we can verify that the above 2-step process of creating a `GateGroup` results in an identical `GateGroup` as the `make_geq_gate` function:

In [ ]:
smf.make_geq_gate(nphdd, 30) == ng_nphdd30

What about AND (`&`) and OR (`|`) gates?
Just add more gates to the specification and make the truthtable appropriately larger:

In [ ]:
# and gate truthtable
and_tt = np.array([[False, False],
                   [False, True]])
# gate for one column
gate_nphdd30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphdd,), {'m':30.0, 'vec':np.array([1.0])})
# gate for DIFFERENT column
gate_nphda30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphda,), {'m':30.0, 'vec':np.array([1.0])})
ng_nphdd30_AND_nphda30 = smf.GateGroup(and_tt, gate_nphdd30, gate_nphda30)

And we can verify that this indeed creates an identical gate as the AND of gates made using `make_geq_gate`:

In [ ]:
ng_nphdd30_AND_nphda30 == smf.make_geq_gate(nphdd, 30) & smf.make_geq_gate(nphda, 30)

## GateGroup regularization

When you create a `GateGroup` smfBursts goes through a regularization process.
This regularization converts the arbitrarily specified GateGroup into a "cannonical" form.

This cannonical form means that the `Gate` objects inside are always in a specific order.

So no matter if the user specified `gate1, gate2` or `gate2, gate1`- the resulting object will **always** have the gates in the order `gate1, gate2`.
The necessary transposition of the axes of the truthtable is automatically carried out.

We can demonstrate this below:

Note that in this example, we will define a truthtable that is not symetric, so that we can verify the necessar transposition upon reordering the gates takes place:

In [ ]:
# "subtraction" gate truthtable
sub_tt = np.array([[False, False],
                   [True, False]])
# reverse "subtraction" gate truthtable
inv_sub_tt = np.array([[False, True],
                       [False, False]])
# gate for one column
gate_nphdd30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphdd,), {'m':30.0, 'vec':np.array([1.0])})
# gate for DIFFERENT column
gate_nphda30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphda,), {'m':30.0, 'vec':np.array([1.0])})

# deine 2 gates in different ways that *should* be equivalent:
ng_nphdd30_SUB_nphda30 = smf.GateGroup(sub_tt, gate_nphdd30, gate_nphda30)
ng_nphda30_RSUB_nphdd30 = smf.GateGroup(inv_sub_tt, gate_nphda30, gate_nphdd30) # order of gates reversed, and truthtable reversed

# First, check truthtables are identical, next see gates in same order
np.all(ng_nphdd30_SUB_nphda30 == ng_nphda30_RSUB_nphdd30), [ga == gb for ga, gb in zip(ng_nphdd30_SUB_nphda30.gates, ng_nphda30_RSUB_nphdd30.gates)]

### Truthtable "broadcasting"

When creating a `GateGroup` in the `GateGroup(truthtable, gate0, gate1, ...)` style,
`gate0, ...` can also be other `GateGroup` objects.

During the regularization process, a "broadcasted" truthtable is evaluated.
All combinations of each dimension along the input `truthtable` and the `gateX.truthtable`s are evaluated.
This process also ensures that any identical gates in the different `gateX.gates` attributes are approriately unified.

So we can create an AND gate between two gates by creating a new `GateGroup` directly with
`GateGroup(AND_Truthtable, gateA, gateB)`

In [ ]:
# gate for one column
gate_nphdd30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphdd,), {'m':30.0, 'vec':np.array([1.0])})
# gate for DIFFERENT column
gate_nphda30 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphda,), {'m':30.0, 'vec':np.array([1.0])})

# deine 2 gates in different ways that *should* be equivalent:
ng_nphdd30_SUB_nphda30 = smf.GateGroup(sub_tt, gate_nphdd30, gate_nphda30)
ng_nphda30_RSUB_nphdd30 = smf.GateGroup(inv_sub_tt, gate_nphda30, gate_nphdd30) # order of gates reversed, and truthtable reversed

ng_comb = smf.GateGroup(and_tt, ng_nphdd30_SUB_nphda30, ng_nphda30_RSUB_nphdd30)
print(ng_comb.truthtable)
for i, g in enumerate(ng_comb.gates):
    print(f'----------{i}-----------')
    print(g.description)
    print('-------------------------')

### Gate Comparison

During the regularization process, after broadcasting, a simplification process takes place.
This checks for any gates that can be dropped without changing the result.

In the simplest case considert the following truthtable:

```
[[False, True]
 [False, True]]
GateA, GateB
```
can be simplified to

```
[False, True]
GateB
```

But this regularization goes further- when certain regions of the truthtable will never be evaluated-
for instance consider two greater than gates on the *same* column, 
it is impossible for that gate to evaluate in the position where the gate with the higher min value is true and the lower min value is false.
smfBursts can test truthtables accounting for these blank regions, and simplify accordingly.

The upshot is that
$G_{>40} \cap G_{>50} \rightarrow G_{>50}$
And far more complex behavior.



In [ ]:
gate_nphdd40 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphdd,), {'m':40.0, 'vec':np.array([1.0])})
gate_nphdd50 = smf.Gate(smf.gates.LIN_GEQ_gate, (nphdd,), {'m':50.0, 'vec':np.array([1.0])})

g_nphdd40_AND_50 = smf.GateGroup(and_tt, gate_nphdd40, gate_nphdd50)

In [ ]:
print(g_nphdd40_AND_50.truthtable)
for i, g in enumerate(g_nphdd40_AND_50.gates):
    print(f'----------{i}-----------')
    print(g.description)
    print('-------------------------')

And, indeed, this gate is equivalent if we had just created it with the larger gate:

In [ ]:
g_nphdd40_AND_50 == smf.GateGroup(np.array([False, True]), gate_nphdd50)

What about a gate with a truthtable equivalent to:
$G_{<40} \cap G_{>50} \rightarrow \emptyset$

In [ ]:
g_lnphdd40_AND_nphdd50 = smf.GateGroup(np.array([[False, True],[False, False]]), gate_nphdd40, gate_nphdd50)

In [ ]:
print(g_lnphdd40_AND_nphdd50)

Indeed, it evaluates to a "none" type of gate.

### Gate overlap codes

> **Note**
>
> This section can be considered "implementation details"
> The only time a user needs to know this is if they want to implement their own Gate function

The gate reduction relies on "overlap codes".
Between any 2 gates, only certain combinations of True/False are possible.
In overlap codes, each combination is represented by a specific bit in an integer (a bitmask).

A B     | B False    | B True
--------|------------|------------
A False | 0b0001 = 1 | 0b0100 = 4
A True  | 0b0010 = 2 | 0b1000 = 8

If a given combination is possible, add it to the overlap code.

The classmethod (function called from `GateGroup`, not an instance of `GateGroup`) `GateGroup.overlap`
can be called on 2 gates `GateGroup.overlap(gateA, gateB)` to get the overlap code.

#### Examples for overlap codes

If there are 2 independent gates, for instance based on 2 different columns, then all combinations are possible:

$G_{<10} H_{<10}$   | $H_{<10}$ False  | $H_{<10}$ True
--------------------|------------------|----------------
$G_{<10}$ False     | Possible         | Possible
$G_{<10}$ True      | Possible         | Possible

so the overlap code is 0b1111 = 15

On the other hand, if we have $G_{<10}$ and $G_{<20}$ of the same column

$G_{<10} G_{<20}$ | $G_{<20}$ False  | $G_{<20}$ True
------------------|------------------|----------------
$G_{<10}$ False   | Possible         | Impossible
$G_{<10}$ True    | Possible         | Possible

so the overlap code is 0b1011 = 11

or, what if we had $G_{<10}$ and $G_{>20}$
Then the table of possibilities becomes

$G_{<10} G_{>20}$ | $G_{>20}$ False  | $G_{>20}$ True
------------------|------------------|----------------
$G_{<10}$ False   | Possible         | Possible
$G_{<10}$ True    | Possible         | Impossible

so the overlap code is 0b0111


#### Limitations

The internal implementation (described in creating GateDefs) 
relies on computing overlaps between each combination of `Gate` in the `GateGroup`s.
This requires per-gate comparison functions.
However, a full implementation of all interactions has not been done.
Therefore PhotonBursts errs on the side of caution.
It is generally better to consider a given T/F combination of gates possible even if it is impossible.
This results in non-reduction, 
whereas considering a given combination impossible could result in incorrect gate reduction.

Currently for instance, while it is known that nph_raw columns can never be less than 0,
the evaluation does not take this into account.
Similarly, certain ratio_raw columns are restricted to the interval $[0, 1]$,
but again, the evaluation of the comparison between such columns does not take this into account.
In all these cases the result is an overlap code of 0b1111 when some of the 1's should be 0's.

Also, elliptical gates compared to each other always evaluate as `0b1111`.
This is because the elliptical gate is an N-D ellipse in its definition.
Solving comparison between 2D ellipses is fairly simple, 
but the generalization to n-dimensions is non-trivial, and not yet implemented.
Two pseudo-code algorithms have been published
[Gilitschenski I, Hanebeck U, IEEE 2014](https://doi.org/10.1109/SDF.2014.6954724)
[Calbert J et. al. IFAC 2023](https://doi.org/10.1016/j.ifacol.2023.10.1088),
but these require a different "cannonical form" of the hyperelipsoid,
contributions from anyone who can either implement a new algorithm for the upper diagonal
cannonical form or convert from the upper diagnonal to those used in the above ciations,
are very welcome to be integrated.
Therefore any elliptical gate is always kept when performing gate logical operations.

#### Examples

We can test this on some of our simple gates. 
Note that since overlap codes are bitmasks, it is often easist to understand them by calling 
`bin()` to see the individual bits.

In [ ]:
bin(smf.GateGroup.overlap(g_nphda30, g_nphdd30))

Second gate is "inside" another

In [ ]:
bin(smf.GateGroup.overlap(g_nphda30, g_nphda30_AND_nphdd30))

Gates are inverse of each other

In [ ]:
bin(smf.GateGroup.overlap(g_nphda30, ~g_nphda30))

#### the `in` operation (Generally Useful)

If it is not possible for gateA to be True when gateB is False, gateA is a subset of gateB.
This would be bit 2, so if `GateGroup.overlap` returns 0bxx0x (where x can be 0 or 1), then gateA is a subset of gateB.

Since this is a very useful test, you can use the `in` python keyword to test if one `GateGroup` is a subset of another.

In [ ]:
g_nphda30 in g_nphdd30, g_nphda30_AND_nphdd30 in g_nphdd30

## GateDef: building your own gate function

PhotonBursts allows defining additional Gate functions through GateDef objects.

To define a Gate function, the following must be done:

**1. Define gate function**

This function should take numpy array(s) as positional argument(s), and gate parameters as keyword arguments.

When gating, PhotonBursts hands the function the arrays of each column in the gate as \*args to the gate func, 
followed by the params of the gate as \*\*kwargs

The function should return a boolean mask, of the same size as the rows of the columns.

**2. Define gate regularization and/or verification functions (if necessary)**

There are 2 potential function that can be defined.
Whether they should be defined depends on the nature of the gate function you are creating they are:

1. Regularize
    - function that converts params into canonical form.
    - should be defined if params have multiple equivalent representations
      eg in an N-D linear gate, the best representation of $\vec{V} \cdot \vec{C} >= m$,
      where $\vec{V}$ is a vector orthogonal to the line of division, $\vec{C}$ is the
      vector of all columns, and $m$ is a magnitude.
      Equivalent gates are possible by varying the magnitudes of $\vec{V}$ inverse
      proportionally to $m$. Therefor regularization should take place to ensure $\vec{V}$ is normal,
      and $m$ is positive. In the case of our geq_cube column, this is not necessary
    - the regularize function should have the signature `regularize(params:dict, colorder:tuple[int,...])->dict` or `regularize(params:dict)->dict`
      depending on if the columns need to be re-ordered or not (sortcol argument to GateDef).
      When a new gate is created the params will be handed to the regularize function before creating the gate.
      The actual params set will be the dictionary that the regularize function returns.
        - colorder identifies the sorted order of the columns relative to the input.
          The definition allows the following transformation of vectors that match the dimensions of columns:
          `old_order[sort] = new_order` ie place the index of `sort[i]` of the old order the into index `i` of the new order.
2. Verify
    - function that verifies the gate is valid. Use this to throw errors on non-sense specifications
    - called at very end of column creation process
    - takes signature `verify(columns:tuple[smf.Column,...], params:smf.tupledict])`
    - should only raise an error if the combinatio of columns and params is invalid for the given gate function

It should be noted that errors can be raised through regularize, and thus if regularization takes place, 
so even if non-sense gates are possible, if regularization is needed, it is often easier to handle both through regularize.

**3. register functions**

PhotonBursts requires functions used in a number of cases to be "registered".
This adds them to in internal dictionary of functions known as PyCode functions.
The gate function, as well as (if used) the regularize and verify functions must be specified.

To register a function simply call `smf.datamodel.immutabledata.register_PyCode(func)`

**4. Additional parameters to specify**

There are 2 more options that can be specified:

1. nparents : a 2 element array specifying the minimum number of columns and the maximum number of columns
2. sortcol : boolean, should be true if equivalent gates can be defined with different orderings of columns. If this is true, a regularization function almost certainly needs to be defined
3. atomic : usually true, whether or not the gate operates like a ufunc ie the value of each row is only dependant on the values of the columns in that same row.
 

> **Note**
>
> Usually new `GateDef`s should be defined not in a jupyter notebook, but in an imported module
> This is because in a key step, the function needs to be registered.
> While possible in a jupyter notebook, it is not ideal, as if a cell is re-run or modified
> a new function may get the same name, as a registered function, and the original lost.
> This makes repeating the operation difficult.
> Further, once a function is registed, it is constant for the duration of the kernel.


See the 2 examples below for defining rather strange gates:

### Example: A Quadratic gate

Lets setup all the definitions we need:

In [ ]:
# A simple gate function
def geq_square(columna:np.ndarray[np.float64], columnb:np.ndarray[np.float64], 
               a:float, b:float, c:float)->np.ndarray[np.bool_]:
    return a*(columna**2)+ b*columna + c < columnb

# register function as PyCode with PhotonBursts
# Note that this can only be done once per python kernel- 
# this prevents confusion of overwriting a function with another of the same name
smf.datamodel.immutabledata.register_PyCode(geq_square)

square_param = smf.tupledict(('a', float), ('b', float), ('c', float))

# This function demonstrates the idea, but does nothing
def verify_square(columns, params):
    assert params['a'] != 0, "Gate is equivalent to a linear gate"
# register validate function, beware of same warnings as registering geq_square
smf.datamodel.immutabledata.register_PyCode(verify_square)

GEQ_square = smf.datamodel.GateDef(geq_square, square_param, 
                                          nparents=np.array([2,2]), verify=verify_square)

And lets see a simple, dummy demonstration of this gate:

In [ ]:
gate_square = smf.Gate(GEQ_square, (nphdd, nphda), {'a':0.01, 'b':-1.4, 'c':50.0})

g_square = smf.GateGroup(np.array([False, True]), gate_square)
ng_square = ~g_square

smf.plot.scatter(data, nphdd, nphda, gate=g_square, s=1.0)
smf.plot.scatter(data, nphdd, nphda, gate=ng_square, s=1.0)

# lets see the line between them:
x = np.linspace(0,200,200)
y = gate_square.params['a']*x**2 + gate_square.params['b']*x+gate_square.params['c']
plt.plot(x,y, c='k')

So far, nothing has been set to determine how to compare the GEQ_square gate,
we can make another gate that is all "inside" the other, but gate reduction will not take place

In [ ]:
gate_square2 = smf.Gate(GEQ_square, (nphdd, nphda), {'a':0.01, 'b':-1.4, 'c':60.0})

g_square2 = smf.GateGroup(np.array([False, True]), gate_square2)

smf.plot.scatter(data, nphdd, nphda, gate=g_square, s=2.0)
smf.plot.scatter(data, nphdd, nphda, gate=g_square2, s=2.0)
g_square_AND_square2 = g_square & g_square2

In [ ]:
print(g_square_AND_square2.truthtable)
for i, g in enumerate(g_square_AND_square2.gates):
    print(f'-------{i}-------')
    print(g.description)
    print('-----------------')

Now, no gate comparison function has been set.
This means that two gates with the same columns and same a and b parameters will
still not reduce, even though they should.

To solve this, we use the 
`smf.datamodel.immutabledata.GateDefinition.set_gate_comparison` function
It takes 3 arguments: `gatedefA, gatedefB, func)`

`gatedefA` and `gatedefB` are two `GateDef` objects, (note that the same pair can only be registered once).
`func` is a callable, it should accept a function call of `func(gateA, gateB)` where `gateA` and `gateB` are
gates that use the `gatedefA` and `gatedefB` functions respectively.
The return value needs to be an overlap code (int between 1 and 16 inclusive).

**Note** that it is the responsibility of the func to check that the columns match.

**Be Careful** if an overlap function returns a value with 0 bit where a 1 should be, the result will be incorrect, over-zealous reductions.
It costs time if a gate is not reduced when in can be, but your result is wrong if a gate is reduced when in should not have been.

> **Caution**
>
> This section is for demonstration ONLY.
> You should set all gate comparisons before creating any gates, preferably in an imported module.
> This is because setting the gate comparison after creating a compound gategroup
> you run the risk of changing how reduction works, which leads to the potential for
> the same logical operation creating different gategroups before and after setting.
> (exactly what happens in this demonstration example)

In [ ]:
def square_compare(gateA:smf.Gate, gateB:smf.Gate)->int:
    # columns are identical
    if gateA.columns == gateB.columns:
        # quick catch for identical gates- logic path simplified later if we exclude this
        if gateA.params == gateB.params:
            return 0b1001
        # determine if intersection- if b^2-4ac >= 0, gates intersect
        deltaA = gateA.params['a'] - gateB.params['a']
        deltaB = gateA.params['b'] - gateB.params['b']
        deltaC = gateA.params['c'] - gateB.params['c']
        if deltaB**2 - 4*deltaA*deltaC < 0.0:
            return 0b1111
        # we know that the intersection of parabolas is in imaginary plane
        # now determine which T/F combination is impossible
        # first determine direction of parabolas- postive or negative
        # then compare c values to see which is in which
        if gateA.params['a'] > 0:
            if deltaC == 0.0:
                return 0b1101 if deltaA > 0.0 else 0b1011
            return 0b1011 if deltaC > 0.0 else 0b1101
        # it should be noted that the logic is just reversign the middle 2 bits
        else:
            if deltaC == 0.0:
                return 0b1011 if deltaA > 0.0 else 0b1101
            return 0b1101 if deltaC > 0.0 else 0b1011
    # case of orthogonally pointing 
    elif gateA.columns == gateB.colums[::-1]:
        # Solving for whether these intersect is left as an exercise for the user
        pass
    return 0b1111

smf.datamodel.tables.GateDefinition.set_gate_comparison(GEQ_square, GEQ_square, square_compare)

# it would also be possible to create a gate comparison with the LIN_GEQ_gate
# this will also be left to the user to write

# def lin_square_compare(lingate, squaregate):
#    ... # comparison logic
# smf.datamodel.tables.GateDefinition.set_gate_comparison(smf.datamodel.gates.LIN_GEQ_gate, GEQ_square, lin_square_compare)

In [ ]:
g_square_AND_square2 = g_square & g_square2
print(g_square_AND_square2.truthtable)
for i, g in enumerate(g_square_AND_square2.gates):
    print(f'-------{i}-------')
    print(g.description)
    print('-----------------')

Below is another example where the nubmer of columns is flexible, 
and it uses a regularization function instead of verify.

In [ ]:
def abs_geq_gate(*columns:np.ndarray, vec:np.ndarray[np.float64], m:float)->np.ndarray[np.bool_]:
    return np.sum(np.array(np.abs(columns)).T*vec, axis=1) < mn

smf.datamodel.immutabledata.register_PyCode(abs_geq_gate)

def regularize_abs_geq(params:dict, colorder:tuple[int,...])->dict:
    if 'vec' not in params:
        raise ValueError("must specify vec for abs_geq_gate")
    # ensure vec is correct shape
    vec = np.atleast_1d(params['vec'])
    # check vec is 1d
    if vec.ndim != 1:
        raise ValueError(f"vec must be 1d array, (input is {vec.ndim}")
    # size fo vec does not equal number of columns
    elif vec.size != len(colorder):
        raise ValueError(f"vec must be same size as columns, got {vec.shape[0]}, expected {len(colorder)}")
    # makes it so that m has a "default" of 0.0
    m = float(params.get('m', 0.0))
    # colorder is given as a tuple, which will be interpreted as set of indexes
    # casting to array allows us to simply re-order array of vec
    colorder = np.array(colorder, dtype=np.int64)
    # reorder vec to match col order
    vec = vec[colorder]
    # compute normalization factor for vec and m
    norm = np.sqrt(np.sum(vec**2))
    vec /= norm # normalize vec- now vec is 
    m /= norm
    if (err:=np.argwhere(vec == 0.0)).shape[0]:
        raise ValueError(f"element {err[0,0]} of vec to small, is 0.0 after normalization")
    if vec[0] < 0.0:
        vec, m = -vec, -m
    return dict(vec=vec, m=m)

smf.datamodel.immutabledata.register_PyCode(regularize_abs_geq)

abs_geq_params = smf.tupledict(('vec', np.ndarray), ('m', float))
ABS_GEQ_gate = smf.datamodel.GateDef(abs_geq_gate, abs_geq_params, sortcol=True, regularize=regularize_abs_geq)

# writing gate comparisons left to user